### ETL-пайплайн: Почасовые метеоданные Open-Meteo в PostgreSQL

Описание задачи:
Ноутбук реализует полный цикл извлечения, трансформации и загрузки исторических метеорологических наблюдений из публичного REST API Open-Meteo в аналитическую схему `raw_weather.hourly_observations` базы данных PostgreSQL 15.

- Источник данных: Open-Meteo Historical Weather API (`https://archive-api.open-meteo.com/v1/archive`)
- Целевая таблица: `raw_weather.hourly_observations`
- Идемпотентность: `INSERT ... ON CONFLICT (city, measured_at) DO UPDATE`

In [32]:
import logging
import os
import time
from datetime import datetime, timedelta

import pandas as pd
import psycopg2
import requests
from dotenv import load_dotenv
from psycopg2.extras import execute_values

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("weather_etl")
logger.info("Библиотеки импортированы")

2026-11-19 21:45:25 | INFO     | Библиотеки импортированы


In [33]:
load_dotenv(dotenv_path="../.env")

DB_CONFIG = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": int(os.getenv("POSTGRES_PORT", 5432)),
    "dbname": os.getenv("POSTGRES_DB", "weather_dw"),
    "user": os.getenv("POSTGRES_USER", "etl_user"),
    "password": os.getenv("POSTGRES_PASSWORD")
}

In [34]:
API_BASE_URL = os.getenv("API_BASE_URL")
START_DATE = os.getenv("START_DATE")
END_DATE = os.getenv("END_DATE")

CITIES = [
    {"city": "Moscow", "lat": 55.7558, "lon": 37.6173},
    {"city": "Vienna", "lat": 48.2082, "lon": 16.3738},
    {"city": "Budapest", "lat": 47.4979, "lon": 19.0402},
    {"city": "Zurich", "lat": 47.3769, "lon": 8.5417},
    {"city": "Bergen", "lat": 60.3913, "lon": 5.3221},
]

logger.info(f"Период: с {START_DATE} по {END_DATE}, городов: {len(CITIES)}")

2026-11-19 21:45:25 | INFO     | Период: с 2025-01-01 по 2025-12-31, городов: 5


In [35]:
start_dt = datetime.strptime(START_DATE, "%Y-%m-%d").date()
end_dt = datetime.strptime(END_DATE, "%Y-%m-%d").date()

In [36]:
date_chunks = []
cur_dt = start_dt
while cur_dt <= end_dt:
    chunk_end = min(cur_dt + timedelta(days=30), end_dt)
    date_chunks.append((cur_dt.isoformat(), chunk_end.isoformat()))
    cur_dt = chunk_end + timedelta(days=1)

In [37]:
raw_responses = []

with requests.Session() as session:
    for city_info in CITIES:
        for chunk_start, chunk_end in date_chunks:
            params = {
                "latitude": city_info["lat"],
                "longitude": city_info["lon"],
                "start_date": chunk_start,
                "end_date": chunk_end,
                "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
                "timezone": "UTC",
            }
            try:
                resp = session.get(API_BASE_URL, params=params, timeout=15)
                resp.raise_for_status()
                payload = resp.json()
                payload["_meta_city"] = city_info["city"]
                raw_responses.append(payload)
                logger.info(f"Extract OK: {city_info['city']} [{chunk_start} .. {chunk_end}]")
                time.sleep(0.2)
            except requests.RequestException as exc:
                logger.error(f"Ошибка запроса для {city_info['city']} ({chunk_start}..{chunk_end}): {exc}")
                raise

logger.info(f"Всего получено API-страниц: {len(raw_responses)}")

2026-11-19 21:45:25 | INFO     | Extract OK: Moscow [2025-01-01 .. 2025-01-31]
2026-11-19 21:45:26 | INFO     | Extract OK: Moscow [2025-02-01 .. 2025-03-03]
2026-11-19 21:45:26 | INFO     | Extract OK: Moscow [2025-03-04 .. 2025-04-03]
2026-11-19 21:45:26 | INFO     | Extract OK: Moscow [2025-04-04 .. 2025-05-04]
2026-11-19 21:45:26 | INFO     | Extract OK: Moscow [2025-05-05 .. 2025-06-04]
2026-11-19 21:45:27 | INFO     | Extract OK: Moscow [2025-06-05 .. 2025-07-05]
2026-11-19 21:45:27 | INFO     | Extract OK: Moscow [2025-07-06 .. 2025-08-05]
2026-11-19 21:45:27 | INFO     | Extract OK: Moscow [2025-08-06 .. 2025-09-05]
2026-11-19 21:45:27 | INFO     | Extract OK: Moscow [2025-09-06 .. 2025-10-06]
2026-11-19 21:45:28 | INFO     | Extract OK: Moscow [2025-10-07 .. 2025-11-06]
2026-11-19 21:45:28 | INFO     | Extract OK: Moscow [2025-11-07 .. 2025-12-07]
2026-11-19 21:45:28 | INFO     | Extract OK: Moscow [2025-12-08 .. 2025-12-31]
2026-11-19 21:45:28 | INFO     | Extract OK: Vienna 

In [38]:
frames = []
for item in raw_responses:
    hourly = item.get("hourly", {})
    if not hourly or "time" not in hourly:
        continue

    df_chunk = pd.DataFrame({
        "city": item["_meta_city"],
        "latitude": round(float(item["latitude"]), 4),
        "longitude": round(float(item["longitude"]), 4),
        "measured_at": pd.to_datetime(hourly["time"]),
        "temperature_c": pd.to_numeric(hourly["temperature_2m"], errors="coerce"),
        "humidity_pct": pd.to_numeric(hourly["relative_humidity_2m"], errors="coerce"),
        "precipitation_mm": pd.to_numeric(hourly["precipitation"], errors="coerce"),
        "wind_speed_kmh": pd.to_numeric(hourly["wind_speed_10m"], errors="coerce"),
    })
    frames.append(df_chunk)

In [39]:
clean_df = pd.concat(frames, ignore_index=True)

before_drop = len(clean_df)
clean_df = clean_df.dropna(subset=["temperature_c", "humidity_pct"]).copy()
dropped = before_drop - len(clean_df)

logger.info(f"Transform: удалено {dropped} строк с пропусками в temperature_c или humidity_pct")

2026-11-19 21:45:41 | INFO     | Transform: удалено 0 строк с пропусками в temperature_c или humidity_pct


In [40]:
clean_df = clean_df.drop_duplicates(subset=["city", "measured_at"], keep="last")

vapor_pressure = (clean_df["humidity_pct"] / 100.0) * 6.105 * (
        2.71828 ** ((17.27 * clean_df["temperature_c"]) / (237.7 + clean_df["temperature_c"]))
)
wind_ms = clean_df["wind_speed_kmh"] / 3.6
clean_df["heat_index_c"] = (clean_df["temperature_c"] + 0.33 * vapor_pressure - 0.70 * wind_ms - 4.00).round(2)

for col in ["temperature_c", "humidity_pct", "precipitation_mm", "wind_speed_kmh"]:
    clean_df[col] = clean_df[col].round(2)

logger.info(f"Transform завершён. Итого строк к загрузке: {len(clean_df)}")
clean_df.head()

2026-11-19 21:45:41 | INFO     | Transform завершён. Итого строк к загрузке: 43800


,city,latitude,longitude,measured_at,temperature_c,humidity_pct,precipitation_mm,wind_speed_kmh,heat_index_c
0,Moscow,55.7821,37.5764,2025-01-01 00:00:00,-3.5,85,0.0,13.5,-8.80
1,Moscow,55.7821,37.5764,2025-01-01 01:00:00,-4.2,84,0.0,12.9,-9.47
2,Moscow,55.7821,37.5764,2025-01-01 02:00:00,-4.7,86,0.0,12.5,-9.91
3,Moscow,55.7821,37.5764,2025-01-01 03:00:00,-5.4,87,0.0,10.8,-10.33
4,Moscow,55.7821,37.5764,2025-01-01 04:00:00,-6.2,89,0.0,10.6,-11.13


In [41]:
UPSERT_SQL = """
             INSERT INTO raw_weather.hourly_observations (city, latitude, longitude, measured_at,
                                                          temperature_c, humidity_pct, precipitation_mm, wind_speed_kmh,
                                                          heat_index_c)
             VALUES %s ON CONFLICT (city, measured_at) DO
             UPDATE SET
                 temperature_c = EXCLUDED.temperature_c,
                 humidity_pct = EXCLUDED.humidity_pct,
                 precipitation_mm = EXCLUDED.precipitation_mm,
                 wind_speed_kmh = EXCLUDED.wind_speed_kmh,
                 heat_index_c = EXCLUDED.heat_index_c,
                 loaded_at = CURRENT_TIMESTAMP; \
             """

In [42]:
records = [
    (
        row.city,
        float(row.latitude),
        float(row.longitude),
        row.measured_at.to_pydatetime(),
        float(row.temperature_c),
        float(row.humidity_pct),
        float(row.precipitation_mm),
        float(row.wind_speed_kmh),
        float(row.heat_index_c),
    )
    for row in clean_df.itertuples(index=False)
]

In [43]:
conn = None
try:
    conn = psycopg2.connect(**DB_CONFIG)
    with conn:
        with conn.cursor() as cur:
            execute_values(cur, UPSERT_SQL, records, page_size=1000)
    logger.info(f"Load успешно завершён: обработано {len(records)} записей (UPSERT).")
except psycopg2.Error as exc:
    logger.error(f"Ошибка при записи в PostgreSQL: {exc}")
    raise
finally:
    if conn is not None:
        conn.close()

2026-11-19 21:45:43 | INFO     | Load успешно завершён: обработано 43800 записей (UPSERT).


In [44]:
conn = None
try:
    conn = psycopg2.connect(**DB_CONFIG)
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM raw_weather.hourly_observations;")
        total_rows = cur.fetchone()[0]
        logger.info(f"Всего строк в таблице raw_weather.hourly_observations: {total_rows}\n")

        cur.execute("""
                    SELECT city,
                           COUNT(*)                     AS rows_cnt,
                           ROUND(AVG(temperature_c), 2) AS avg_temp_c,
                           ROUND(AVG(heat_index_c), 2)  AS avg_comfort_c
                    FROM raw_weather.hourly_observations
                    GROUP BY city
                    ORDER BY city;
                    """)
        summary_df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
        logger.info("Сводка по городам:")
        display(summary_df)

        cur.execute("""
                    SELECT city,
                           measured_at,
                           temperature_c,
                           humidity_pct,
                           precipitation_mm,
                           wind_speed_kmh,
                           heat_index_c
                    FROM raw_weather.hourly_observations
                    ORDER BY measured_at ASC, city ASC LIMIT 5;
                    """)
        sample_df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
        logger.info("\nПервые 5 записей в БД:")
        display(sample_df)
except psycopg2.Error as exc:
    logger.error(f"Ошибка при проверке данных в PostgreSQL: {exc}")
    raise
finally:
    if conn is not None:
        conn.close()

2026-11-19 21:45:44 | INFO     | Всего строк в таблице raw_weather.hourly_observations: 43800

2026-11-19 21:45:44 | INFO     | Сводка по городам:


,city,rows_cnt,avg_temp_c,avg_comfort_c
0,Bergen,8760,7.88,4.73
1,Budapest,8760,12.50,10.27
2,Moscow,8760,8.16,5.21
3,Vienna,8760,11.53,8.86
4,Zurich,8760,10.40,8.88


2026-11-19 21:45:44 | INFO     | 
Первые 5 записей в БД:


,city,measured_at,temperature_c,humidity_pct,precipitation_mm,wind_speed_kmh,heat_index_c
0,Bergen,2025-01-01,-1.60,85.00,0.10,9.10,-5.85
1,Budapest,2025-01-01,-1.80,96.00,0.00,4.00,-4.88
2,Moscow,2025-01-01,-3.50,85.00,0.00,13.50,-8.80
3,Vienna,2025-01-01,-4.20,94.00,0.00,7.20,-8.21
4,Zurich,2025-01-01,-1.90,99.00,0.00,2.60,-4.67
